In [13]:
import pandas as pd

In [14]:
ratings = pd.read_csv('rating.csv')

In [15]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [16]:
movies = pd.read_csv('movie.csv')

In [17]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [9]:
from sklearn.preprocessing import LabelEncoder

user_encoder = LabelEncoder()

In [25]:
ratings['user'] = user_encoder.fit_transform(ratings['userId'])

In [26]:
ratings

,userId,movieId,rating,timestamp,user
0,1,2,3.5,2005-04-02 23:53:47,0
1,1,29,3.5,2005-04-02 23:31:16,0
2,1,32,3.5,2005-04-02 23:33:39,0
3,1,47,3.5,2005-04-02 23:32:07,0
4,1,50,3.5,2005-04-02 23:29:40,0
...,...,...,...,...,...
20000258,138493,68954,4.5,2009-11-13 15:42:00,138492
20000259,138493,69526,4.5,2009-12-03 18:31:48,138492
20000260,138493,69644,3.0,2009-12-07 18:10:57,138492
20000261,138493,70286,5.0,2009-11-13 15:42:24,138492


In [32]:
ratings['user'].value_counts()

user
118204    9254
8404      7515
82417     5646
121534    5520
125793    5491
          ... 
3029        20
3033        20
36559       20
3045        20
127907      20
Name: count, Length: 138493, dtype: int64

In [34]:
movies['item'] = user_encoder.fit_transform(movies['movieId'])

In [35]:
movies

,movieId,title,genres,item
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1
2,3,Grumpier Old Men (1995),Comedy|Romance,2
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,3
4,5,Father of the Bride Part II (1995),Comedy,4
...,...,...,...,...
27273,131254,Kein Bund für's Leben (2007),Comedy,27273
27274,131256,"Feuer, Eis & Dosenbier (2002)",Comedy,27274
27275,131258,The Pirates (2014),Adventure,27275
27276,131260,Rentun Ruusu (2001),(no genres listed),27276


In [36]:
movies['item'].value_counts()

item
27277    1
0        1
1        1
2        1
3        1
        ..
18       1
17       1
16       1
15       1
14       1
Name: count, Length: 27278, dtype: int64

In [39]:
import pandas as pd
import numpy as np

movies = pd.read_csv('movie.csv')       # movieId, title, genres
ratings = pd.read_csv('rating.csv')     # userId, movieId, rating

# Compute popularity stats per movie
stats = ratings.groupby('movieId').agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

movies = movies.merge(stats, on='movieId', how='left')
movies['avg_rating'] = movies['avg_rating'].fillna(0)
movies['rating_count'] = movies['rating_count'].fillna(0)

# Extract year from title e.g. "Toy Story (1995)"
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['clean_title'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)

In [46]:
movies

,movieId,title,genres,avg_rating,rating_count,year,clean_title,search_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.921240,49695.0,1995,Toy Story,Toy Story Adventure Animation Children Comedy ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.211977,22243.0,1995,Jumanji,Jumanji Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,3.151040,12735.0,1995,Grumpier Old Men,Grumpier Old Men Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.861393,2756.0,1995,Waiting to Exhale,Waiting to Exhale Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,3.064592,12161.0,1995,Father of the Bride Part II,Father of the Bride Part II Comedy
...,...,...,...,...,...,...,...,...
27273,131254,Kein Bund für's Leben (2007),Comedy,4.000000,1.0,2007,Kein Bund für's Leben,Kein Bund für's Leben Comedy
27274,131256,"Feuer, Eis & Dosenbier (2002)",Comedy,4.000000,1.0,2002,"Feuer, Eis & Dosenbier","Feuer, Eis & Dosenbier Comedy"
27275,131258,The Pirates (2014),Adventure,2.500000,1.0,2014,The Pirates,The Pirates Adventure
27276,131260,Rentun Ruusu (2001),(no genres listed),3.000000,1.0,2001,Rentun Ruusu,Rentun Ruusu (no genres listed)


In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine title + genres as search content
movies['search_text'] = movies['clean_title'] + ' ' + movies['genres'].str.replace('|', ' ')

vectorizer = TfidfVectorizer(ngram_range=(1, 2), analyzer='word')
tfidf_matrix = vectorizer.fit_transform(movies['search_text'])

def search(query, top_n=10, genre_filter=None):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    movies['score'] = scores
    results = movies[movies['score'] > 0.01].copy()

    # Optional genre filter
    if genre_filter:
        results = results[results['genres'].str.contains(genre_filter, case=False)]

    # Rank by TF-IDF score + boost popular/highly rated movies
    results['final_score'] = (
        results['score'] * 0.6 +
        (results['avg_rating'] / 5) * 0.25 +
        (np.log1p(results['rating_count']) / 10) * 0.15
    )

    return results.sort_values('final_score', ascending=False)[
        ['title', 'genres', 'avg_rating', 'rating_count', 'final_score']
    ].head(top_n)

In [42]:
# Trie-based prefix autocomplete
class TrieNode:
    def __init__(self):
        self.children = {}
        self.movies = []  # store (popularity_score, title, movieId)

class AutoComplete:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, title, movie_id, popularity):
        node = self.root
        for char in title.lower():
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
            node.movies.append((popularity, title, movie_id))

    def search(self, prefix, top_n=5):
        node = self.root
        for char in prefix.lower():
            if char not in node.children:
                return []
            node = node.children[char]
        # Sort by popularity descending
        sorted_movies = sorted(node.movies, key=lambda x: -x[0])
        seen, results = set(), []
        for _, title, mid in sorted_movies:
            if mid not in seen:
                seen.add(mid)
                results.append({'movieId': mid, 'title': title})
            if len(results) == top_n:
                break
        return results

# Build the autocomplete index
ac = AutoComplete()
for _, row in movies.iterrows():
    popularity = row['avg_rating'] * np.log1p(row['rating_count'])
    ac.insert(row['clean_title'], row['movieId'], popularity)

# Usage
 # → Inception, Incredible, Incognito...

In [47]:
ac.search("no") 

[{'movieId': 908, 'title': 'North by Northwest'},
 {'movieId': 55820, 'title': 'No Country for Old Men'},
 {'movieId': 930, 'title': 'Notorious'},
 {'movieId': 2671, 'title': 'Notting Hill'},
 {'movieId': 1348,
  'title': 'Nosferatu (Nosferatu, eine Symphonie des Grauens)'}]